In [1]:
import os
import json
import google.generativeai as genai
from kscLLM.index import ROOT_PATH
from kscLLM.util import to_markdown, get_model


GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print("set") if GOOGLE_API_KEY else print("unset")
genai.configure(api_key=GOOGLE_API_KEY)

set


/home/lukas/Programming/uni/threatintel-showcase/llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = get_model()

# Load collected logs

In [3]:
with open(ROOT_PATH / "tmp/stix.json", "r") as file:
    observables_bundle = json.load(file)

prompt = f"""Report malicious activity in the following STIX Domain Objects. 
A credential access attack is searched. First a token is fetched. This token is then loaded in the application by calling the metadata.google.internal service-accounts token endpoint through a SSRF attack during image upload.
Find for each step of this attack all related bundles.

Respond in this format for each malicious SDO:

Artifact ID: <artifact id>
Message: <message of the bundle>
Explanation: <explanation why this SDO is malicious>

{str(observables_bundle)}"""


response_indicator = model.generate_content(prompt)
# to_markdown(response_indicator.text)
print(response_indicator.text)

Artifact ID: artifact--1521dc2d-197e-5ad1-86b5-268cd4947d9f
Message: I1006 20:23:41.615857   15901 metadata.go:209] [conn-id:70f74499e0a9b5ae rpc-id:51e504fe025f3a63 remote-addr:10.1.3.10:48816 pod:kube-system/konnectivity-agent-9ff757d86-n4j2l] Sending passthrough request to GCE metadata server: endpoint: /computeMetadata/v1/instance/service-accounts/default/token
Explanation: This log indicates a Server-Side Request Forgery (SSRF) attack, where an attacker is exploiting the application to fetch a token from the metadata service, likely with the goal of gaining unauthorized access. 

Artifact ID: artifact--1a672ab4-2bd7-5bcb-92d3-300b24d1057c
Message: I1006 20:23:46.510991   15906 metadata.go:209] [conn-id:1f4f0d89db1764d3 rpc-id:364e3fbdf62f719e remote-addr:10.1.3.6:41438 pod:kube-system/kube-dns-89d659d4f-t6m2x] Sending passthrough request to GCE metadata server: endpoint: /computeMetadata/v1/instance/service-accounts/default/token
Explanation:  Similar to the previous artifact, thi

In [4]:
true_positive_bundle_ids = [
    "bundle--060e9e7f-1c2f-4d1c-8f60-20308669c4f3", # Fetched token for pod
    "bundle--66c85317-a94d-495d-838a-982b82332c9b", # google.internal image upload call
]

true_positive_bundles = [bundle for bundle in observables_bundle if bundle["id"] in true_positive_bundle_ids]
true_positive_bundles

[]

In [5]:
with open(ROOT_PATH / "tmp/positives.json", "w") as file:
    json.dump(true_positive_bundles, file)